In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    doc["module"] = doc["filename"].split("/")[0]
    documents.append(doc)

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

To do

1. The documents have been downloaded in memory.

1. Index them so that they can be searchable.

1. Ask questions about the documents using an LLM.

### Q1. How many lesson pages

How many lesson pages are in the dataset?
- 24
- 72
- 240
- 720

Answer: 72

In [5]:
len(files)

72

### Q2. Indexing and searching

Answer: 01-agentic-rag/lessons/14-agentic-loop.md

In [27]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    
    index.fit(documents)
    return index

index = build_index(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[:2]

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [26]:
for i in search_results:
    print(i["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


### Q3. RAG

Answer: 7000

In [7]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [8]:
from dotenv import load_dotenv

load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

It keeps calling the model inside a `while True` loop, then checks whether the model returned any `function_call` items.

- If there is a function call, it runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is: **no function calls this turn**.


In [11]:
response.usage.input_tokens

7121

### Q4. Chunking

Answer: 295

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [14]:
len(chunks)

295

### Q5. RAG with chunking

Answer: 3× fewer

In [ ]:
index = build_index(chunks)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

In [17]:
response.usage.input_tokens

2304

### Q6. Turning it into an agent

Answer:3 to 4 times

In [20]:
# !uv add toyaikit

In [21]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

import json

In [22]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lesson contents to answer the user's query.
    """
    return index.search(
        query,
        num_results=5
    )

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [24]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': "Search the course lesson contents to answer the user's query.",
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
instructions = """You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
